1(loading the dataframe

In [281]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
df=pd.read_csv("/content/AmesHousing.csv")


2)checking the shape

In [282]:
def preview_data(df):
    #first 5 rows
    print("first 5 rows\n",df.head(5))
    #shape
    print("shape of the df",df.shape)

3)inspect data types


In [283]:
def info_before(df):
    #info
    print("info before fixing\n",df.info())

4)fixing data types

In [284]:
def fix_types(df):
    if "Roof Style" in df.columns:
        df["Roof Style"]=df["Roof Style"].str.replace(r"[^a-z\s]","",regex=True).str.strip().str.lower()
    df["Lot Area"]=pd.to_numeric(df["Lot Area"],errors="coerce")
    df["Year Built"]=pd.to_datetime(df["Year Built"],errors="coerce")
    return df

5)info after fixing types

In [285]:
def info_after(df):#fixed info
    print("info after fixing\n",df.info())

6)handling and fixing data types

In [286]:
def handle_missingnes(df):
    #missing values
    missing=df.isnull().sum().sort_values(ascending=False)
    print("Missing values:\n", missing[missing>0])
    """columns such as presented contain over 80-99% missing values.
    these columns likely represent features that are absent in most
    houses rather than true missing data"""
    df=df.drop(columns=["Pool QC","Misc Feature","Alley","Fence"])
    #fill numric with median
    """missing values in numeric columns
     were filled using the median becausee it is robust
      to outliers and preserves the distribution of the data"""
    num_cols=df.select_dtypes(include=["number"]).columns
    for col in num_cols:
        if df[col].isnull().sum()>0:
            df[col]=df[col].fillna(df[col].median())
    #fill categorical with mode
    """missing values in categorical columns were filled using the mode
    since it represents the most frequent category"""

    cat_cols=df.select_dtypes(include=["object",'string','category']).columns
    for col in cat_cols:
        if df[col].isnull().sum()>0:
            df.loc[:,col] =df[col].fillna(df[col].mode()[0])
    return df


7)handling and removing duplicates

In [287]:
def handle_duplicate(df):
    #duplicates
    print("duplicates",df.duplicated().sum())
    df=df.drop_duplicates()
    return df

8)spoting and detecting outliers


In [288]:
def detect_outliers_using_IQR(df):
    #outliers using IQR
    q1, q3 = df["Lot Area"].quantile(0.25),df["Lot Area"].quantile(0.75)
    iqr = q3 - q1
    lower, upper =q1-1.5*iqr,q3+1.5*iqr
    outliers= df[(df["Lot Area"] < lower) | (df["Lot Area"] > upper)]
    print("Number of outliers:", len(outliers))
    cap_99=df["Lot Area"].quantile(0.99)
    df["Lot Area"]=df["Lot Area"].clip(upper=cap_99)
    return df,lower,upper,outliers

9)funal chickups

In [289]:
def final_chicks(df):
    # no duplicates
    dup_count=df.duplicated().sum()
    print("duplicates after cleaning",int(dup_count))
    if dup_count==0:
        print("duplicate rows found")
    #no missing values left
    total_nulls=df.isnull().sum().sum()
    print("total remaining nulls:",int(total_nulls))
    if total_nulls==0:
        print("there are no missing values in the dataset!")
    #all target_vlues>0
    negative_or_zero=(df["Lot Area"]<=0).sum()
    print("target<=0 count:",negative_or_zero)
    if negative_or_zero==0:
        print("target values are all non-nigative")

10)cleaned data function

In [290]:
import os

def clean_data(df):
    preview_data(df)
    info_before(df)
    df=fix_types(df)
    info_after(df)
    df=handle_missingnes(df)
    df=handle_duplicate(df)
    df,lower,upper,outliers=detect_outliers_using_IQR(df)
    final_chicks(df)
    return df
df_cleaned=clean_data(df)
df.to_csv("cleaned_data.csv")

first 5 rows
    Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Alley Lot Shape Land Contour  ... Pool Area Pool QC  Fence Misc Feature  \
0   NaN       IR1          Lvl  ...         0     NaN    NaN          NaN   
1   NaN       Reg          Lvl  ...         0     NaN  MnPrv          NaN   
2   NaN       IR1          Lvl  ...         0     NaN    NaN         Gar2   
3   NaN       Reg          Lvl  ...         0     NaN    NaN          NaN   
4   NaN       IR1          Lvl  ...         0     NaN  MnPrv          NaN   

  Misc Val Mo Sold Yr Sold Sale Type  Sale Condition  SalePr